# MMCT Image Query Pipeline

This notebook demonstrates how to use the refactored **Image Pipeline** within the MMCTAgent framework. The Image Pipeline leverages the **Multi-Modal Critical Thinking (MMCT)** architecture, featuring a collaborative agent team:

- **Planner**: Breaks down queries, selects tools, and synthesizes answers.
- **Critic (Optional)**: Refines the planner's response through feedback loops.

### 🛠️ Key Features
- **Modular Tools**: OCR, Object Detection, Recognition, and ViT (Vision Transformer).
- **Provider System**: Seamless switching between Azure OpenAI, Anthropic, or local models.
- **Streaming**: Real-time feedback via AutoGen's `Console` or async generators.

### 1. Setup and Imports

First, we apply `nest_asyncio` to allow async execution in Jupyter and import the unified pipeline components.

In [ ]:
import os
import nest_asyncio
import asyncio
from mmct.image_pipeline import ImageAgent, ImageQnaTools
from mmct.image_pipeline.config import ImageAgentProviderConfig
from config.provider_config import get_llm_provider

nest_asyncio.apply()

# Ensure environment variables are loaded (endpoint, api_key, etc.)
print("✅ Imports successful")

### 2. Provider Configuration

We use the centralized `get_llm_provider()` helper to hydrate settings from your `.env` file. This creates an `ImageAgentProviderConfig` bundle required by the agent.

In [ ]:
# Initialize provider from environment defaults
llm_provider = get_llm_provider()
provider_config = ImageAgentProviderConfig(llm_provider=llm_provider)

print(f"🚀 Provider initialized: {llm_provider.__class__.__name__}")

### 3. Basic Execution

Run a standard query against a local image. We'll specify a subset of tools to optimize performance.

In [ ]:
# Example image found in the repository
image_path = "./media/uniform_frames/frame_000048.jpg"
query = "What objects are visible in this image? List them clearly."

image_agent = ImageAgent(
    image_path=image_path,
    query=query,
    provider=provider_config,
    use_critic_agent=True,
    tools=[ImageQnaTools.vit, ImageQnaTools.object_detection],
    use_console=True # Set to True to see agent thoughts in console
)

print(f"🔍 Querying image: {image_path}")
response = await image_agent()

print(f"\n✅ Final Response:\n{response.response}")
print(f"\n📊 Usage: {response.tokens}")

### 4. Streaming Mode

You can enable streaming to watch the Planner and Critic interact in real-time. Setting `use_console=True` will use the AutoGen `Console` UI for a rich experience.

In [ ]:
stream_agent = ImageAgent(
    image_path="./media/uniform_frames/frame_000005.jpg",
    query="Can you read any text in this image?",
    provider=provider_config,
    use_critic_agent=True,
    tools=[ImageQnaTools.ocr, ImageQnaTools.vit],
    stream=True,
    use_console=True
)

print("🌊 Starting Streamed Query...\n")
result = await stream_agent()
print(f"\nFinal Answer: {result.response}")

### 5. Custom Provider Example (Anthropic)

The Image Pipeline is provider-agnostic. You can easily implement a custom provider by inheriting from `BaseLLMProvider`. Below is an example for Anthropic Claude.

In [ ]:
from mmct.providers.base import BaseLLMProvider
from typing import Dict, Any, List, Optional

class AnthropicLLMProvider(BaseLLMProvider):
    """Example provider for Anthropic Claude models."""
    def __init__(self, api_key: str, model_name: str = "claude-3-5-sonnet-20241022"):
        self.api_key = api_key
        self.model_name = model_name
        # Initialize your SDK client here

    async def chat_completion(self, messages: List[Dict], **kwargs) -> Dict[str, Any]:
        # Implement API call logic
        pass

    def get_autogen_client(self, **kwargs):
        # Return an autogen-compatible client wrapper
        from autogen_ext.models.anthropic import AnthropicChatCompletionClient
        return AnthropicChatCompletionClient(model=self.model_name, api_key=self.api_key)

# Usage with custom provider:
# custom_provider = ImageAgentProviderConfig(llm_provider=AnthropicLLMProvider(api_key="..."))
print("🛠️ Custom Provider Template Ready")